## 02 — TextBlob Sentiment Analysis

### What changed from the original notebook?

The original analysis ran TextBlob on heavily preprocessed text (`cleaned_reviews`) 
which involved lemmatization, stopword removal, and spell-checking across multiple 
English dictionaries (US, UK, Canadian, Australian, NZ, South African variants), 
plus language detection and Google translation of Spanish/French reviews.

**In this version, we run TextBlob on raw reviews deliberately.** Here's why:

### 1. Consistency with Transformer Analysis (Notebook 03)
Notebook 03 runs the RoBERTa transformer model on raw review text. Since Notebook 04 
compares TextBlob vs Transformer results, both models must receive identical input. 
If TextBlob runs on cleaned text and the transformer runs on raw text, any disagreements 
in Notebook 04 could be caused by preprocessing differences rather than model differences 
— making the comparison analytically invalid.

### 2. Sampling Change
The original notebook sampled ~50K reviews per borough from listings with >100 reviews.
This version uses pre-built samples (~15K reviews per borough) from listings with >200 
reviews, saved in `data/samples/`. These samples are shared across notebooks 02, 03, 
and 04 to ensure all analyses run on identical reviews.

### 3. Neutral Class
The original notebook used a binary classification:
- `polarity >= 0` → Positive  
- `polarity < 0` → Negative

This meant nearly all reviews were classified as Positive (since TextBlob returns 0.0 
for reviews it can't score, which mapped to Positive). 

This version uses a ±0.05 buffer to create a genuine Neutral class:
- `polarity > 0.05` → Positive
- `polarity < -0.05` → Negative
- `-0.05 <= polarity <= 0.05` → Neutral

This makes TextBlob's output comparable to the transformer's three-class output, 
and better captures Airbnb's well-documented "grade inflation" problem where guests 
write lukewarm but polite reviews.

In [1]:
# Install if needed
!pip install textblob --quiet


You should consider upgrading via the 'C:\Users\avant\source\airbnb-reviews-nlp\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from pathlib import Path


# Path setup
ROOT        = Path.cwd().parent
SAMPLES_DIR = ROOT / "data" / "samples"

#### Load data from data/samples

In [3]:
manhattan_sample_df = pd.read_csv(SAMPLES_DIR / "manhattan_sample.csv")
brooklyn_sample_df  = pd.read_csv(SAMPLES_DIR / "brooklyn_sample.csv")

print(f"Manhattan sample: {len(manhattan_sample_df):,} reviews")
print(f"Brooklyn sample : {len(brooklyn_sample_df):,} reviews")
print(f"\nColumns: {manhattan_sample_df.columns.tolist()}")

Manhattan sample: 14,946 reviews
Brooklyn sample : 14,652 reviews

Columns: ['listing_id', 'review_id', 'date', 'reviews', 'neighbourhood_group', 'room_type', 'price', 'review_length']


### TextBlob Analysis



TextBlob returns two scores per review:
- **Polarity** — sentiment score between -1.0 (most negative) and 1.0 (most positive)
- **Subjectivity** — between 0.0 (objective/factual) and 1.0 (subjective/opinionated)

We retain both as they serve different purposes:
- Polarity → sentiment label for comparison with transformer in Notebook 04
- Subjectivity → additional signal for host recommendations in Notebook 05

Check to see if there are any null reviews etc. 

In [6]:

for name, df in [('Manhattan', manhattan_sample_df), ('Brooklyn', brooklyn_sample_df)]:
    print(f"\n{name}")
    print(f"  Total rows       : {len(df):,}")
    print(f"  Null reviews     : {df['reviews'].isna().sum()}")
    print(f"  Empty strings    : {(df['reviews'].astype(str).str.strip() == '').sum()}")
    print(f"  'nan' strings    : {(df['reviews'].astype(str) == 'nan').sum()}")


Manhattan
  Total rows       : 14,946
  Null reviews     : 0
  Empty strings    : 0
  'nan' strings    : 0

Brooklyn
  Total rows       : 14,652
  Null reviews     : 0
  Empty strings    : 0
  'nan' strings    : 0


In [ ]:
def get_textblob_sentiment(text):
    """Returns polarity and subjectivity scores for a review."""
    analysis = TextBlob(str(text))
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity


In [5]:
def categorize_sentiment(polarity):
    """
    Three-class sentiment label using ±0.05 buffer.
    
    TextBlob returns 0.0 for reviews it cannot score, which would all map 
    to Positive under binary classification. The buffer creates a genuine 
    Neutral class that captures lukewarm reviews — particularly important 
    for Airbnb where guests are socially conditioned to write politely 
    even when dissatisfied ('the apartment was fine', 'decent location').
    
    This matches the three-class output of the transformer model 
    (Positive / Neutral / Negative), enabling direct comparison in Notebook 04.
    """
    if polarity > 0.05:
        return "Positive"
    elif polarity < -0.05:
        return "Negative"
    else:
        return "Neutral"


def categorize_subjectivity(subjectivity):
    """Three-class subjectivity label."""
    if subjectivity < 0.3:
        return "Objective"
    elif subjectivity <= 0.7:
        return "Mixed"
    else:
        return "Subjective"

In [ ]:
import time

def run_textblob_analysis(df, borough_name, text_col='reviews'):
    """
    Runs TextBlob sentiment analysis on a borough sample.
    
    Args:
        df          : Borough sample DataFrame
        borough_name: For logging
        text_col    : Column containing raw review text
    
    Returns:
        DataFrame with polarity, subjectivity, and label columns added
    """
    
    print(f"\n{'='*50}")
    print(f"Running TextBlob analysis: {borough_name}")
    print(f"{'='*50}")
    print(f"Reviews: {len(df):,}")
    
    df = df.copy().reset_index(drop=True)
    
    start = time.time()
    
    # Score all reviews
    scores = df[text_col].apply(
        lambda x: pd.Series(get_textblob_sentiment(x), 
                            index=['polarity', 'subjectivity'])
    )
    df = pd.concat([df, scores], axis=1)
    
    elapsed = time.time() - start
    print(f"Scoring complete in {elapsed:.1f}s ({elapsed/len(df)*1000:.1f}ms per review)")
    
    # Apply labels
    df['sentiment_label']    = df['polarity'].apply(categorize_sentiment)
    df['subjectivity_label'] = df['subjectivity'].apply(categorize_subjectivity)
    
    # Summary
    # print(f"\nSentiment distribution:")
    # print(df['sentiment_label'].value_counts(normalize=True).round(3))
    # print(f"\nMean polarity    : {df['polarity'].mean():.4f}")
    # print(f"Mean subjectivity: {df['subjectivity'].mean():.4f}")
    
    return df

### Run Textblob Analysis

In [8]:
manhattan_textblob = run_textblob_analysis(manhattan_sample_df, 'Manhattan')
brooklyn_textblob  = run_textblob_analysis(brooklyn_sample_df,  'Brooklyn')


Running TextBlob analysis: Manhattan
Reviews: 14,946
Scoring complete in 16.3s (1.1ms per review)

Sentiment distribution:
sentiment_label
Positive    0.866
Neutral     0.119
Negative    0.014
Name: proportion, dtype: float64

Mean polarity    : 0.3792
Mean subjectivity: 0.5712

Running TextBlob analysis: Brooklyn
Reviews: 14,652
Scoring complete in 15.5s (1.1ms per review)

Sentiment distribution:
sentiment_label
Positive    0.909
Neutral     0.082
Negative    0.009
Name: proportion, dtype: float64

Mean polarity    : 0.3991
Mean subjectivity: 0.5983


In [14]:
print(manhattan_textblob['sentiment_label'].value_counts(normalize=True).round(3))
print(f"\nMean polarity    : {manhattan_textblob['polarity'].mean():.4f}")
print(f"Mean subjectivity: {manhattan_textblob['subjectivity'].mean():.4f}")

sentiment_label
Positive    0.866
Neutral     0.119
Negative    0.014
Name: proportion, dtype: float64

Mean polarity    : 0.3792
Mean subjectivity: 0.5712


Overall for Manhattan, texblob analysis reveals **86.6% Positive reviews, 1.4% Negative and 11.9% Neutral reviews. The mean polarity is around 0.38 and mean subjectivity 0.57** 

In [13]:
print(brooklyn_textblob['sentiment_label'].value_counts(normalize=True).round(3))
print(f"\nMean polarity    : {brooklyn_textblob['polarity'].mean():.4f}")
print(f"Mean subjectivity: {brooklyn_textblob['subjectivity'].mean():.4f}")

sentiment_label
Positive    0.909
Neutral     0.082
Negative    0.009
Name: proportion, dtype: float64

Mean polarity    : 0.3991
Mean subjectivity: 0.5983


For Brooklyn :  textblob indicates there are **90.9% Positive, 0.9% Negative and 8.2% Neutral reviews. The mean polarity is ~0.4 and mean subjectivity is at ~0.6**

<u>Across both boroughs:</u> Polarity of around 0.4 on -1 to +1 scale indicates that the reviews are positive but not ecstatically or overwhelmingly so. This is further held up by the ~0.6 subjectivity score - suggesting the reviews are more opinions than facts. This aligns, again, with the nature of the text being analysed ("reviews") being based on personal experiences. 

### Save the results

In [9]:
manhattan_textblob.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length,polarity,subjectivity,sentiment_label,subjectivity_label
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59,0.343758,0.645455,Positive,Mixed
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143,0.303431,0.573268,Positive,Mixed
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11,0.800000,0.900000,Positive,Subjective
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92,0.397500,0.573333,Positive,Mixed
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186,0.264613,0.454656,Positive,Mixed


In [10]:
brooklyn_textblob.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length,polarity,subjectivity,sentiment_label,subjectivity_label
0,7097,40929473,2015-08-03,Great location!,Brooklyn,Private room,205.0,2,1.000000,0.750000,Positive,Subjective
1,7097,1163859806775409442,2024-05-24,"Jane se montre très disponible, logement très ...",Brooklyn,Private room,205.0,28,0.000000,0.000000,Neutral,Objective
2,7097,761535232,2021-05-23,Loved this place.. the apartment and garden we...,Brooklyn,Private room,205.0,40,0.511667,0.766667,Positive,Subjective
3,7097,8907441,2013-11-25,Airbnb\r<br/>\r<br/>Fantastic place for enjoyi...,Brooklyn,Private room,205.0,220,0.145977,0.441320,Positive,Mixed
4,7097,1287058306615520248,2024-11-10,Ms. Jane was very accommodating and very atten...,Brooklyn,Private room,205.0,38,0.099394,0.704924,Positive,Subjective


In [11]:
#RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = ROOT / "data" / "results"
manhattan_textblob.to_csv(RESULTS_DIR / "manhattan_textblob.csv", index=False)
brooklyn_textblob.to_csv(RESULTS_DIR  / "brooklyn_textblob.csv",  index=False)

In [12]:

print("Manhattan sample output:")
print(manhattan_textblob[['listing_id', 'reviews', 'polarity', 
                           'subjectivity', 'sentiment_label', 
                           'subjectivity_label']].head())

print("\nBrooklyn sample output:")
print(brooklyn_textblob[['listing_id', 'reviews', 'polarity',
                          'subjectivity', 'sentiment_label',
                          'subjectivity_label']].head())


Manhattan sample output:
   listing_id                                            reviews  polarity  \
0        6990  Cynthia is a very nice host , hospitable and s...  0.343758   
1        6990  What can I say about Cynthia?\r<br/>We stayed ...  0.303431   
2        6990  Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...  0.800000   
3        6990  In one word - WOW! I really had the time of my...  0.397500   
4        6990  Being a first time Airbnb user, I had a hard t...  0.264613   

   subjectivity sentiment_label subjectivity_label  
0      0.645455        Positive              Mixed  
1      0.573268        Positive              Mixed  
2      0.900000        Positive         Subjective  
3      0.573333        Positive              Mixed  
4      0.454656        Positive              Mixed  

Brooklyn sample output:
   listing_id                                            reviews  polarity  \
0        7097                                    Great location!  1.000000   
1      